# Financial Fraud Data Cleaning and Analysis 
This notebook focuses on cleaning and analyzing the `fraud_detection_v4.csv` dataset.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv("../Datasets/fraud_detection_v4.csv")
df.head()

,step,customer,age,gender,zipcodeOri,merchant,zipMerchant,category,amount,fraud
0,0,'C1093826151','4','M','28007','M348934600','28007','es_transportation',4.55,0
1,0,'C352968107','2','M','28007','M348934600','28007','es_transportation',39.68,0
2,0,'C2054744914','4','F','28007','M1823072687','28007','es_transportation',26.89,0
3,0,'C1760612790','3','M','28007','M348934600','28007','es_transportation',17.25,0
4,0,'C757503768','5','M','28007','M348934600','28007','es_transportation',35.72,0


## 1. Data Cleaning

In [3]:
# Check for missing values
print("Missing values:\n", df.isnull().sum())

# Check for duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Remove single quotes from string columns
string_cols = ['customer', 'age', 'gender', 'zipcodeOri', 'merchant', 'zipMerchant', 'category']
for col in string_cols:
    if col in df.columns:
        df[col] = df[col].str.replace("'", "")

# Handle 'age' column (it might contain 'U' for Unknown or other codes)
print("\nUnique values in Age:", df['age'].unique())

# Convert Age to numeric if possible, or leave as categorical if codes are used
# For this dataset, age is often coded as 0-6 and 'U'

# Drop columns that usually have single values in this dataset (ZIP codes)
cols_to_drop = ['zipcodeOri', 'zipMerchant']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print("\nCleaned Data Info:")
df.info()

Missing values:
 step           0
customer       0
age            0
gender         0
zipcodeOri     0
merchant       0
zipMerchant    0
category       0
amount         0
fraud          0
dtype: int64

Duplicate rows: 0

Unique values in Age: <StringArray>
['4', '2', '3', '5', '1', '6', 'U', '0']
Length: 8, dtype: str

Cleaned Data Info:
<class 'pandas.DataFrame'>
RangeIndex: 594643 entries, 0 to 594642
Data columns (total 8 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   step      594643 non-null  int64  
 1   customer  594643 non-null  str    
 2   age       594643 non-null  str    
 3   gender    594643 non-null  str    
 4   merchant  594643 non-null  str    
 5   category  594643 non-null  str    
 6   amount    594643 non-null  float64
 7   fraud     594643 non-null  int64  
dtypes: float64(1), int64(2), str(5)
memory usage: 36.3 MB


## 2. Data Analysis

In [ ]:
# Statistical summary
print("Descriptive Statistics:\n", df.describe())

# Fraud Distribution
fraud_counts = df['fraud'].value_counts()
fraud_percent = df['fraud'].value_counts(normalize=True) * 100
print(f"\nFraud counts:\n{fraud_counts}")
print(f"\nFraud percentages:\n{fraud_percent}")

In [ ]:
# Analysis by Category
category_analysis = df.groupby('category').agg({
    'amount': 'mean',
    'fraud': 'sum',
    'customer': 'count'
}).rename(columns={'customer': 'transaction_count'})

category_analysis['fraud_rate'] = (category_analysis['fraud'] / category_analysis['transaction_count']) * 100
print("Category Analysis (Sorted by Fraud Rate):\n", category_analysis.sort_values(by='fraud_rate', ascending=False))

In [ ]:
# Transaction amounts: Fraud vs Non-Fraud
print("\nAmount summary by Fraud status:")
print(df.groupby('fraud')['amount'].describe())

### Initial Insights:
- The dataset is highly imbalanced (typical for fraud detection).
- Certain categories like 'leisure' and 'travel' often show much higher fraud rates in this synthetic dataset.
- Fraudulent transactions typically have a significantly higher average amount than legitimate ones.